<a href="https://colab.research.google.com/github/weirygon/IA/blob/main/MEI_IA_Lojas_MM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INICIALIZANDO O GEMINI**

---



In [ ]:
!pip install -U google-generativeai
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

ReadTimeout: HTTPConnectionPool(host='localhost', port=39141): Read timed out. (read timeout=60.0)

In [ ]:
model = genai.GenerativeModel('gemini-pro')

Alterando filtos de segurança

In [ ]:
safety_settings = [
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE" # Exemplo: Bloquear discurso de ódio de média probabilidade e acima
    },
    {
        "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
        "threshold": "BLOCK_NONE"
    },
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE"
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE"
    }
]

**Teste conexão com o Gimini**

In [ ]:
response = model.generate_content("Gemini voce esta ai?")
print("Resposta: ", response.text)

# **ABRINDO ARQUIVOS DE CONVERSAS**

---



In [ ]:
file_path = '/content/Lojas MM Jul 1000.csv'

with open(file_path, 'r', encoding='utf-8', errors='replace') as file:
    rows = file.readlines()

    for i, row in enumerate(rows):
      rows[i] = row.split(';')

## Criando Classes

In [ ]:
class Mensagem:
  def __init__(self, texto, autor, data):
    self.texto = texto
    self.autor = autor
    self.data = data

  def __str__(self):
    return f"{self.data} {self.autor}: {self.texto}"

class Cobrador:
  def __init__(self, nome):
    self.nome = nome
    self.conversas = list()



## Colocando mensagems no dicionario

In [ ]:
from datetime import datetime

mensagens = []
conversas = dict()

for row in rows[1:]:
  mensagem = Mensagem(row[7], row[6], datetime.strptime(row[10], "%Y-%m-%d %H:%M:%S.%f"))
  mensagens.append(mensagem)

  if conversas.get(row[0]) == None:
    conversas[row[0]] = [mensagem]
  else:
    conversas[row[0]].append(mensagem)


## Ordenando conversas por data

In [ ]:
for id in conversas:
  conversas[id].sort(key=lambda x: x.data)

## Adicionando conversas em Colaborador

In [ ]:
cobradores = list()
nomes_colaboradores = list()

for id in conversas:
  for row in rows:
    if row[0] == id:
      if row[5] not in nomes_colaboradores:
        nomes_colaboradores.append(row[5])
        cobradores.append(Cobrador(row[5]))

      for cobrador in cobradores:
        if cobrador.nome == row[5]:

          cobrador.conversas.append({id: conversas[id]})
          break

      break

for cobrador in cobradores:
  print("==================\n" + cobrador.nome)
  for conversa in cobrador.conversas:
    for id in conversa:
      print("\n",id, "++++++++++++")
      for msg in conversa[id]:
        print(msg)


## Removendo dados sensiveis
- CPF

In [ ]:
import re

# Regex para encontrar CPFs nos formatos:
# - Com pontuação (123.456.789-00)
# - Sem pontuação (12345678900)
# - Com espaços (123 456 789 00)
padrao_cpf = r'\b\d{3}[\.\s]?\d{3}[\.\s]?\d{3}[-\s]?\d{2}\b'

for cobrador in cobradores:
  for conversa in cobrador.conversas:
    for id in conversa:
      for msg in conversa[id]:
        msg.texto = re.sub(padrao_cpf, 'XXX.XXX.XXX-XX', msg.texto)

* Arquivos

In [ ]:
for cobrador in cobradores:
  for conversa in cobrador.conversas:
    for id in conversa:
      for msg in conversa[id]:
        if "application/pdf" in msg.texto:
          msg.texto = "Boleto.pdf"
        elif "audio/mpeg" in msg.texto:
          msg.texto = "audio.mpeg"
        elif "audio/ogg" in msg.texto:
          msg.texto = "audio.ogg"
        elif "image/gif" in msg.texto:
          msg.texto = "imagem.gif"
        elif "image/jpeg" in msg.texto:
          msg.texto = "imagem.jpeg"
        elif "image/png" in msg.texto:
          msg.texto = "imagem.png"

## Imprimindo conversas

In [ ]:
for i, id in enumerate(conversas):
  print("======================\n", i, id, type(conversas[id]))
  for msg in conversas[id]:
      print(msg)

# **ANALISE DAS CONVERSAS**

Foi enviados as conversas ao Gemini para calular a porcentagem de sucesso na negociação seguindo os seguintes critérios:

## Critérios para Avaliação da Probabilidade de Pagamento:

### Menções a Pagamento:
* 100%: O usuário confirma o pagamento com detalhes (data, valor, comprovante).
* 75-90%: O usuário se compromete firmemente a pagar em uma data específica, demonstra conhecer o valor devido e pede informações para pagamento.
* 50-70%: O usuário expressa intenção de pagar, mas demonstra dúvidas sobre o valor, solicita mais informações ou propõe datas de pagamento mais distantes.
* 25-49%: O usuário reconhece a dívida, mas apresenta objeções (falta de dinheiro, problemas financeiros, etc.) ou tenta negociar valores muito abaixo do devido.
* 0-24%: O usuário nega a dívida, ignora as mensagens, demonstra hostilidade ou encerra a conversa abruptamente.
### Negociação de Valores/Prazos:
* Alta Probabilidade (75-100%): O usuário aceita as condições propostas pelo atendente ou chega a um acordo razoável.
* Média Probabilidade (50-74%): O usuário tenta negociar insistentemente, mas demonstra abertura para chegar a um acordo.
* Baixa Probabilidade (0-49%): O usuário se mostra inflexível e não aceita as propostas do atendente.
### Tom da Conversa:
* Positivo/Colaborativo (75-100%): O usuário se comunica de forma educada, demonstra interesse em resolver a situação e responde às mensagens prontamente.
* Neutro (50-74%): O usuário mantém uma comunicação formal, sem demonstrar muita empolgação ou resistência.
* Negativo/Agressivo (0-49%): O usuário se comunica de forma grosseira, ignora as mensagens ou demonstra hostilidade.
### Ações Concretas:
* 100%: O usuário envia comprovante de pagamento.
* 75-90%: O usuário solicita código de barras, link para pagamento ou outras informações necessárias para efetuar o pagamento.
* 0-49%: O usuário apenas conversa, mas não toma nenhuma ação concreta em direção ao pagamento.

## **PREPARANDO PROMT**

In [ ]:
texto = ""
for msg in conversas['53b5946a-2b14-4865-b4d3-01891c6bed4a']:
  texto = texto + str(msg) + "\n"

  prompt = f"Analise SOMENTE essa conversa de cobrança e retorne APENAS um número inteiro entre 0 e 100 e uma texto extremamente pequeno justivicando a porcentagem, retorne o texto sem qualquer acento da utilizado da lingua portuguesa e se houver a frase 'audio.ogg' ou 'audio.mpeg' coloque no fim o texto a frase 'CONTEM AUDIO', ao fim do texto de explicação adicione a frase 'COTEM AUDIO'. Separe esses 3 fatoes de saida por ';'. Não inclua nenhuma outra informação, texto ou explicação \n Considere os seguintes critérios para a análise: \n * Menções a Pagamento: Intenção, compromisso, solicitação de informações para pagamento.\n * Negociação: Ofertas, contrapropostas, concordância com valores e prazos.\n * Tom da Conversa: Positivo, neutro ou negativo.\n * Ações Concretas: Solicitação de boletos, links de pagamento, confirmação de pagamento.\n\nExemplos:\nConversa 1 (Alta Probabilidade): 'Ok, quero pagar com Pix. Me manda o código?'\nSaída: 90\nConversa 2 (Baixa Probabilidade): 'Vou ver o que posso fazer. Depois eu entro em contato.'\nSaída: 20\n\n{texto}"

print(prompt)

response = model.generate_content(
              prompt,
              safety_settings=safety_settings
          )
print("Resposta: ", response.text)

Analise SOMENTE essa conversa de cobrança e retorne APENAS um número inteiro entre 0 e 100 e uma texto extremamente pequeno justivicando a porcentagem, retorne o texto sem qualquer acento da utilizado da lingua portuguesa e se houver a frase 'audio.ogg' ou 'audio.mpeg' coloque no fim o texto a frase 'CONTEM AUDIO', ao fim do texto de explicação adicione a frase 'COTEM AUDIO'. Separe esses 3 fatoes de saida por ';'. Não inclua nenhuma outra informação, texto ou explicação 
 Considere os seguintes critérios para a análise: 
 * Menções a Pagamento: Intenção, compromisso, solicitação de informações para pagamento.
 * Negociação: Ofertas, contrapropostas, concordância com valores e prazos.
 * Tom da Conversa: Positivo, neutro ou negativo.
 * Ações Concretas: Solicitação de boletos, links de pagamento, confirmação de pagamento.

Exemplos:
Conversa 1 (Alta Probabilidade): 'Ok, quero pagar com Pix. Me manda o código?'
Saída: 90
Conversa 2 (Baixa Probabilidade): 'Vou ver o que posso fazer. Depo

## Criando arquivo de saida

In [ ]:
import csv

date = datetime.now()

with open(f'/content/MMFiltred 1000 {date}.csv', 'w', newline='') as file:
    writer = csv.writer(file, delimiter=';')

    # Cabeçalho
    writer.writerow(["TickedId", "Cobrador", "DataInicio", "DataFim", "Sucesso", "MotivoSucesso"])

In [ ]:
import time

with open(f'/content/MMFiltred 1000 {date}.csv', 'a', newline='') as file:
    writer = csv.writer(file, delimiter=';')

    texto = ""

    for cobrador in cobradores:
      for conversa in cobrador.conversas:
        for i, id in enumerate(conversa):
          for msg in conversa[id]:
            texto = texto + str(msg) + "\n"

          prompt = f"Analise SOMENTE essa conversa de cobrança e retorne APENAS um número inteiro entre 0 e 100 e uma texto extremamente pequeno justivicando a porcentagem, retorne o texto sem qualquer acento da utilizado da lingua portuguesa e se houver a frase 'audio.ogg' ou 'audio.mpeg' coloque no fim o texto a frase 'CONTEM AUDIO', ao fim do texto de explicação adicione a frase 'COTEM AUDIO'. Separe esses 3 fatoes de saida por ';'. Não inclua nenhuma outra informação, texto ou explicação \n Considere os seguintes critérios para a análise: \n * Menções a Pagamento: Intenção, compromisso, solicitação de informações para pagamento.\n * Negociação: Ofertas, contrapropostas, concordância com valores e prazos.\n * Tom da Conversa: Positivo, neutro ou negativo.\n * Ações Concretas: Solicitação de boletos, links de pagamento, confirmação de pagamento.\n\nExemplos:\nConversa 1 (Alta Probabilidade): 'Ok, quero pagar com Pix. Me manda o código?'\nSaída: 90\nConversa 2 (Baixa Probabilidade): 'Vou ver o que posso fazer. Depois eu entro em contato.'\nSaída: 20\n\n{texto}"
          response = model.generate_content(
              prompt,
              safety_settings=safety_settings
          )
          aux = response.text.split(';')

          writer.writerow([id, cobrador.nome, conversa[id][0].data, conversa[id][-1].data, aux[0], aux[1]])
          print("OK:", id, i)

          time.sleep(20)



OK: 08b8deae-70bc-4115-a716-01894a006d2d 0
OK: 242c4d1a-274f-4b48-9771-01894a178506 0
OK: 5b851dcc-7907-40c3-8b22-018949fe93c3 0
OK: 5fe20937-d9e1-4b56-ac13-01894a06cddb 0
OK: 6e5725a6-d555-400f-909f-01894a1d69c7 0
OK: 7aa359c4-7a03-4a82-9157-01894a139623 0
OK: 871efe10-c8a1-4a8d-b120-01894a0039a2 0
OK: 5df40595-c5fd-49d9-aff0-01891147a213 0
OK: 6dcf3c54-0652-46e3-9361-0189115a29b7 0
OK: b26163fb-b3a5-4749-a1db-018911787485 0
OK: 23f6cd49-87ff-492a-81f2-0189117958f5 0
OK: 5cf40cee-1470-4961-8358-018911947826 0
OK: 78e26228-fc36-44d7-a4c6-018911953221 0
OK: 5a0edb61-7d05-4025-a2cc-0189119de69c 0


KeyboardInterrupt: 